# 제철 공장 전력 소비 예측 — 시계열 EDA와 모델 (Steel Industry)

- 데이터: 전남 광양 제철공장 15분 간격 전력 (35,040행 · 1년)
- 목표: 전력 사용량(kWh) 예측 — 단일 시계열
- 흐름: 불러오기 → 학습 전 확인(집계·누수) → EDA → 형식 변환 → 학습 → 해석
- 참고: 데이터 소개 data_steel_energy.txt

- 이 데이터의 핵심
  · **집계 방식이 변수마다 다름** (합계 / 평균 / 최빈값)
  · **CO2 누수** (사용량에서 계산된 값)
  · 국내 데이터 (한국전력 요금제와 연결)

## 1. 불러오기 · 컬럼명 정리

- 컬럼명에 마침표·괄호가 섞여 있어 정리 권장
  ('Lagging_Current_Reactive.Power_kVarh', 'CO2(tCO2)')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("Steel_industry_data.csv")
print(df.shape)          # (35040, 11)

# 날짜 파싱 (일/월/년 형식)
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y %H:%M")

# 컬럼명 정리
df.columns = ["date","usage_kwh","lag_reactive","lead_reactive",
              "co2","lag_pf","lead_pf","nsm",
              "week_status","day_of_week","load_type"]
df.head()

## 2. 학습 전 확인

### 2-1. CO2는 누수 변수

- CO2 배출량 = 전력 사용량 × 배출계수
- 즉 타깃에서 계산된 값 → 넣으면 정답을 알려주는 셈
- 상관계수로 확인

In [ ]:
print("usage_kwh vs co2 상관:",
      df["usage_kwh"].corr(df["co2"]).round(3))
# 0.99에 가까움 → 누수

df = df.drop(columns=["co2"])
print("제거 후:", df.shape)

### 2-2. 집계는 변수마다 다르다 (가장 중요)

- 15분 간격 → 일별로 집계할 때 변수 성격에 맞춰야 함
  · 사용량·무효전력 = **합계** (누적량)
  · 역률(%) = **평균** (비율은 더하면 무의미)
  · 요일·주중여부 = **첫 값** (하루 내 동일)
  · 부하유형 = **최빈값** (하루에 여러 유형이 섞임)
- 전부 합계로 하면 "하루 평균 역률 3200%" 같은 값이 나옴

In [ ]:
# 하루에 load_type이 몇 종류 나오는지 확인
print("하루 load_type 종류 최대:",
      df.groupby(df["date"].dt.date)["load_type"].nunique().max())
# 3종 → first가 아니라 최빈값(mode)을 써야 함

In [ ]:
daily = df.set_index("date").resample("D").agg({
    "usage_kwh": "sum",          # 누적량 → 합계
    "lag_reactive": "sum",
    "lead_reactive": "sum",
    "lag_pf": "mean",            # 비율 → 평균
    "lead_pf": "mean",
    "week_status": "first",      # 하루 내 동일
    "day_of_week": "first",
    "load_type": lambda s: s.mode()[0],   # 최빈값
}).reset_index()

print(daily.shape)     # (365, 9)
daily.head()

### 2-3. NSM(자정 이후 초)은 일별 집계에서 제외

- 15분 단위에서는 순환 변수(0시와 24시가 인접)
- 일별로 집계하면 의미가 없어 제외함 (위 agg에 없음)

## 3. 시계열 EDA

### 3-1. 전체 추이

In [ ]:
daily.set_index("date")["usage_kwh"].plot(
    figsize=(12,3), title="daily electricity usage (kWh)")
plt.show()
# 조업 패턴에 따른 변동 확인

### 3-2. 요일 패턴

- 제철소는 주중·주말 조업이 다를 수 있음

In [ ]:
(daily.groupby("day_of_week")["usage_kwh"].mean()
   .reindex(["Monday","Tuesday","Wednesday","Thursday",
             "Friday","Saturday","Sunday"])
   .plot(kind="bar", figsize=(8,3), title="mean usage by weekday"))
plt.show()

## 4. 시계열 형식 변환

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame

ts_df = daily.rename(columns={"date":"timestamp",
                              "usage_kwh":"target"})
ts_df["item_id"] = "steel_plant"    # 시계열 1개

ts = TimeSeriesDataFrame.from_data_frame(
    ts_df, id_column="item_id", timestamp_column="timestamp")
ts = ts.convert_frequency(freq="D")
print("변환 완료:", ts.shape)

## 5. 학습 — 공변량 활용

- 요일·주중여부는 **미래도 아는 값** (달력만 봐도 앎)
- known_covariates로 지정하면 예측에 활용됨 (강의 56p)

In [ ]:
from autogluon.timeseries import TimeSeriesPredictor

prediction_length = 14
train_data, test_data = ts.train_test_split(prediction_length)

predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="target",
    freq="D",
    known_covariates_names=["week_status", "day_of_week"],
).fit(train_data, presets="medium_quality", time_limit=600)

In [ ]:
predictor.leaderboard(test_data)

## 6. 해석

In [ ]:
known = test_data.drop(columns=["target"])
predictions = predictor.predict(train_data, known_covariates=known)

predictor.plot(
    data=test_data,
    predictions=predictions,
    quantile_levels=[0.1, 0.9],
    max_history_length=60,
)
plt.show()

### 6-1. 생각해 볼 점

- 공변량(요일)을 넣은 것이 도움이 되었는가
  → 넣지 않고 학습해 비교해 볼 것
- load_type은 공변량으로 쓸 수 있는가?
  · 사용량과 완전히 대응되지는 않음(누수는 아님)
  · 다만 **미래의 부하 유형을 미리 알 수 있는가**가 관건
- 1년치라 연 주기가 1번뿐 → 계절성 학습에는 부족

## 정리

- CO2 = 누수 변수 (사용량에서 계산) → 제거
- 집계는 변수마다 다름: 합계 / 평균 / 최빈값
  → 이 판단은 AutoGluon이 못 함, 사람이 변수 의미를 알아야
- 요일·주중여부 = 미래도 아는 값 → known_covariates
- 1년치라 연 주기 1번 → 계절성 학습엔 부족

- 이 데이터의 교훈: **집계 방식 선택이 곧 전처리**
  → 잘못 집계하면 무의미한 값으로 학습하게 된다